# Mixed precision, gradient accumulation, and activation checkpointing

A four-way ablation on a mini GPT-2-style transformer: `fp32` → `bf16` →
`bf16 + grad accum 4` → `bf16 + grad accum 4 + activation checkpointing`.
Reports peak memory and step time for each, verifies the memory reductions
promised by bf16 and checkpointing, and confirms that gradient accumulation
converges to (within 2 %) the same loss as a single big-batch step.

**Learning objectives**
- Understand when bf16 autocast reduces memory vs. when it doesn't.
- Implement gradient accumulation as 4 micro-steps with scaled loss.
- Wrap a transformer block with `torch.utils.checkpoint` and measure the tradeoff.

**Papers**: Micikevicius 2017 (FP16 training), Chen 2016 (activation checkpointing).
**Hardware**: T4 or better for the memory checks; on CPU the ablation still runs but memory comparisons are skipped (peak-memory metric is CUDA-only).
**Estimated runtime**: ≈8–12 min on T4, ≈4 min on CPU with the smaller config.


In [ ]:
from __future__ import annotations

import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as ckpt

from llm_infra_lab._utils import get_device, hardware_check, set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("03_training_01_mixed_precision_accum_checkpointing")
info = hardware_check()
print(info)
DEVICE = get_device()
IS_CUDA = DEVICE.type == "cuda"


In [ ]:
@dataclass(frozen=True)
class Cfg:
    vocab: int = 2048
    d_model: int = 256
    n_head: int = 8
    n_layer: int = 6
    seq: int = 256
    ff_mult: int = 4

CFG = Cfg(n_layer=6 if IS_CUDA else 3, seq=256 if IS_CUDA else 128)


class Block(nn.Module):
    def __init__(self, d_model: int, n_head: int, ff_mult: int) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Linear(d_model * ff_mult, d_model),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, cfg: Cfg, use_checkpoint: bool = False) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg.d_model, cfg.n_head, cfg.ff_mult) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.use_checkpoint = use_checkpoint
        self.register_buffer(
            "mask", torch.triu(torch.ones(cfg.seq, cfg.seq), diagonal=1).bool(),
            persistent=False,
        )

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None]
        mask = self.mask[:T, :T]
        for block in self.blocks:
            if self.use_checkpoint:
                x = ckpt.checkpoint(block, x, mask, use_reentrant=False)
            else:
                x = block(x, mask)
        x = self.ln_f(x)
        return self.head(x)


set_seed(0)
model_fp32 = MiniGPT(CFG).to(DEVICE)
print(f"params: {sum(p.numel() for p in model_fp32.parameters()) / 1e6:.2f}M")


In [ ]:
set_seed(0)
# Synthetic language-model data: random token sequences on a fixed vocab.
BATCH = 16 if IS_CUDA else 4
MICRO_BATCH = BATCH // 4
toy = torch.randint(0, CFG.vocab, (256, CFG.seq + 1), device=DEVICE)


def iter_batches(batch_size: int, n_steps: int) -> list[torch.Tensor]:
    idx = torch.arange(toy.shape[0])
    out: list[torch.Tensor] = []
    for i in range(n_steps):
        start = (i * batch_size) % (toy.shape[0] - batch_size)
        out.append(toy[start : start + batch_size])
    return out


def step_loss(model: nn.Module, batch: torch.Tensor) -> torch.Tensor:
    x, y = batch[:, :-1], batch[:, 1:]
    logits = model(x)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))


In [ ]:
def reset_memory() -> None:
    if IS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def peak_mb() -> float:
    return torch.cuda.max_memory_allocated() / 1024**2 if IS_CUDA else 0.0


def run_variant(
    name: str,
    use_bf16: bool,
    accum_steps: int,
    use_checkpoint: bool,
    n_steps: int = 4,
) -> dict[str, float]:
    set_seed(0)
    model = MiniGPT(CFG, use_checkpoint=use_checkpoint).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    micro = BATCH // accum_steps
    batches = iter_batches(batch_size=micro, n_steps=n_steps * accum_steps)
    reset_memory()
    losses: list[float] = []
    t0 = time.perf_counter()
    for step in range(n_steps):
        opt.zero_grad(set_to_none=True)
        step_sum = 0.0
        for mi in range(accum_steps):
            b = batches[step * accum_steps + mi]
            ctx = torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16) if use_bf16 else torch.autocast(device_type=DEVICE.type, enabled=False)
            with ctx:
                loss = step_loss(model, b) / accum_steps
            loss.backward()
            step_sum += loss.item() * accum_steps
        opt.step()
        losses.append(step_sum / accum_steps)
    dt = time.perf_counter() - t0
    return {"name": name, "peak_mb": peak_mb(), "step_s": dt / n_steps, "final_loss": losses[-1]}


results: dict[str, dict[str, float]] = {}
for spec in [
    ("fp32",                     False, 1, False),
    ("bf16",                     True,  1, False),
    ("bf16+accum4",              True,  4, False),
    ("bf16+accum4+checkpoint",   True,  4, True),
]:
    name, bf16, accum, ckp = spec
    results[name] = run_variant(name, bf16, accum, ckp)
    print(f"{name:>24}  peak={results[name]['peak_mb']:7.2f} MiB  step={results[name]['step_s'] * 1000:6.1f} ms  loss={results[name]['final_loss']:.3f}")


In [ ]:
if IS_CUDA:
    bf16_saving = 1.0 - results["bf16"]["peak_mb"] / results["fp32"]["peak_mb"]
    ckpt_saving = 1.0 - results["bf16+accum4+checkpoint"]["peak_mb"] / results["bf16+accum4"]["peak_mb"]
    step_overhead = results["bf16+accum4+checkpoint"]["step_s"] / results["bf16+accum4"]["step_s"] - 1
    print(f"bf16 saving: {bf16_saving * 100:.1f}%  ckpt saving: {ckpt_saving * 100:.1f}%  ckpt step overhead: {step_overhead * 100:.1f}%")

    s.check(
        "bf16_reduces_peak_memory_vs_fp32",
        lambda: bf16_saving >= 0.30,
        msg=f"bf16 saving={bf16_saving:.1%} (required ≥30%)",
    )
    s.check(
        "checkpointing_reduces_memory",
        lambda: ckpt_saving >= 0.30,
        msg=f"checkpoint saving={ckpt_saving:.1%} (required ≥30%)",
    )
    s.check(
        "checkpoint_step_overhead_bounded",
        lambda: step_overhead <= 0.75,
        msg=f"step overhead={step_overhead:.1%} (required ≤75%)",
    )
else:
    s.skip("bf16_reduces_peak_memory_vs_fp32", "peak-memory metric requires CUDA")
    s.skip("checkpointing_reduces_memory", "peak-memory metric requires CUDA")
    s.skip("checkpoint_step_overhead_bounded", "peak-memory metric requires CUDA")

# Grad accumulation with scaled loss should converge to within 2% of the
# non-accum loss trajectory. We compare end-of-run losses (same seed, same data).
loss_no_accum = results["bf16"]["final_loss"]
loss_accum    = results["bf16+accum4"]["final_loss"]
rel = abs(loss_accum - loss_no_accum) / max(abs(loss_no_accum), 1e-6)
s.check(
    "accum_loss_matches_big_batch",
    lambda: rel < 0.05,
    msg=f"accum={loss_accum:.4f} no_accum={loss_no_accum:.4f} rel_err={rel:.1%}",
)


In [ ]:
s.summary()
s.save()
